# timm 自定义模型注册示例

本教程演示如何在 timm 中注册一个自定义的简单线性网络模型。

## 目标

我们将创建一个最简单的线性分类器，并将其注册到 timm 的模型注册表中，使其可以通过 `create_model()` 函数创建。

## 1. 导入必要的库

In [ ]:
import torch
import torch.nn as nn
from typing import Optional, Dict, Any

# 导入 timm 的注册相关函数
from timm.models import register_model, generate_default_cfgs
from timm.models._builder import build_model_with_cfg
from timm.models._registry import is_model, list_models
from timm import create_model

## 2. 定义简单的线性网络模型

我们创建一个最简单的线性分类器，包含：
- 一个展平层（Flatten）
- 一个线性层（Linear）

In [ ]:
class SimpleLinearNet(nn.Module):
    """最简单的线性分类网络
    
    Args:
        num_classes: 分类类别数
        in_chans: 输入通道数
        img_size: 输入图像尺寸 (H, W) 或单个整数
        input_size: 完整输入尺寸 (C, H, W)，如果提供则优先使用
        drop_rate: Dropout 比率
    """
    
    def __init__(
        self,
        num_classes: int = 1000,
        in_chans: int = 3,
        img_size: tuple = (224, 224),
        input_size: tuple = None,  # 可选的完整输入尺寸
        drop_rate: float = 0.0,
        **kwargs  # 接收其他参数但不使用
    ):
        super().__init__()
        
        # 处理输入尺寸参数（兼容两种方式）
        if input_size is not None:
            # 如果提供了完整的 input_size，使用它
            self.input_size = input_size
        else:
            # 否则从 in_chans 和 img_size 构建
            if isinstance(img_size, int):
                img_size = (img_size, img_size)
            self.input_size = (in_chans, img_size[0], img_size[1])
        
        self.num_classes = num_classes
        self.drop_rate = drop_rate
        
        # 计算展平后的特征维度
        self.num_features = self.input_size[0] * self.input_size[1] * self.input_size[2]
        
        # 定义网络层
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=drop_rate) if drop_rate > 0 else nn.Identity()
        self.fc = nn.Linear(self.num_features, num_classes)
        
        # 初始化权重
        self._init_weights()
    
    def _init_weights(self):
        """初始化权重"""
        nn.init.xavier_uniform_(self.fc.weight)
        if self.fc.bias is not None:
            nn.init.zeros_(self.fc.bias)
    
    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """特征提取（这里就是展平）"""
        x = self.flatten(x)
        return x
    
    def forward_head(self, x: torch.Tensor, pre_logits: bool = False) -> torch.Tensor:
        """分类头"""
        x = self.dropout(x)
        if pre_logits:
            return x
        x = self.fc(x)
        return x
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """前向传播"""
        x = self.forward_features(x)
        x = self.forward_head(x)
        return x
    
    def get_classifier(self):
        """获取分类器层（timm 标准接口）"""
        return self.fc
    
    def reset_classifier(self, num_classes: int):
        """重置分类器（timm 标准接口）"""
        self.num_classes = num_classes
        self.fc = nn.Linear(self.num_features, num_classes)
        nn.init.xavier_uniform_(self.fc.weight)
        if self.fc.bias is not None:
            nn.init.zeros_(self.fc.bias)

## 3. 定义预训练配置

使用 `generate_default_cfgs` 创建模型的默认配置。

In [ ]:
# 定义配置字典
default_cfgs = generate_default_cfgs({
    # 基础配置（无预训练权重）
    'simple_linear_net': {
        'input_size': (3, 224, 224),
        'num_classes': 1000,
        'mean': (0.485, 0.456, 0.406),
        'std': (0.229, 0.224, 0.225),
        'interpolation': 'bilinear',
        'crop_pct': 0.875,
        'classifier': 'fc',
        'fixed_input_size': True,  # 关键：告诉 build_model_with_cfg 传递 img_size
    },
    # 小尺寸输入配置
    'simple_linear_net.small': {
        'input_size': (3, 32, 32),
        'num_classes': 10,
        'mean': (0.5, 0.5, 0.5),
        'std': (0.5, 0.5, 0.5),
        'interpolation': 'bilinear',
        'crop_pct': 1.0,
        'classifier': 'fc',
        'fixed_input_size': True,  # 关键：告诉 build_model_with_cfg 传递 img_size
    },
})

print("配置已创建:")
for name, cfg in default_cfgs.items():
    print(f"\n模型架构: {name}")
    default_tag = cfg.tags[0] if cfg.tags else 'None'
    print(f"  默认标签: '{default_tag}' {'(无标签)' if default_tag == '' else ''}")
    print(f"  所有可用标签: {[t if t else '(默认)' for t in cfg.tags]}")
    print(f"  配置数量: {len(cfg.cfgs)}")
    for tag, tag_cfg in cfg.cfgs.items():
        tag_display = tag if tag else '(默认)'
        print(f"    - 标签 '{tag_display}': input_size={tag_cfg.input_size}, num_classes={tag_cfg.num_classes}")

## 4. 定义模型创建辅助函数

In [ ]:
def _create_simple_linear_net(variant: str, pretrained: bool = False, **kwargs) -> SimpleLinearNet:
    """创建 SimpleLinearNet 模型的辅助函数
    
    Args:
        variant: 模型变体名称
        pretrained: 是否加载预训练权重
        **kwargs: 传递给模型的其他参数
    
    Returns:
        SimpleLinearNet 模型实例
    """
    return build_model_with_cfg(
        SimpleLinearNet,
        variant,
        pretrained,
        **kwargs,
    )

## 5. 注册模型到 timm

使用 `@register_model` 装饰器注册模型入口函数。

我们将注册两个模型变体：
- `simple_linear_net`: 默认配置（224x224输入，1000类）
- `simple_linear_net_small`: 小尺寸配置（32x32输入，10类）

In [ ]:
# 注册默认配置的模型
@register_model
def simple_linear_net(pretrained: bool = False, **kwargs) -> SimpleLinearNet:
    """Simple Linear Network - 最简单的线性分类器（默认配置）
    
    这是一个演示模型，仅包含展平层和一个线性分类器。
    适合用于理解 timm 的模型注册机制。
    
    默认配置：
    - 输入尺寸: (3, 224, 224)
    - 类别数: 1000
    
    Args:
        pretrained: 是否加载预训练权重（此模型无预训练权重）
        **kwargs: 其他模型参数
    
    Returns:
        SimpleLinearNet 模型实例
    """
    return _create_simple_linear_net('simple_linear_net', pretrained=pretrained, **kwargs)

# # 注册小尺寸配置的模型
# @register_model
# def simple_linear_net_small(pretrained: bool = False, **kwargs) -> SimpleLinearNet:
#     """Simple Linear Network - 小尺寸配置
    
#     小尺寸配置，适合CIFAR-10等小图像数据集。
    
#     小尺寸配置：
#     - 输入尺寸: (3, 32, 32)
#     - 类别数: 10
    
#     Args:
#         pretrained: 是否加载预训练权重（此模型无预训练权重）
#         **kwargs: 其他模型参数
    
#     Returns:
#         SimpleLinearNet 模型实例
#     """
#     # 使用 .small 标签的配置
#     return _create_simple_linear_net('simple_linear_net.small', pretrained=pretrained, **kwargs)

print("✓ 模型 'simple_linear_net' 已成功注册到 timm!")

## 6. 验证模型注册

In [ ]:
# 检查模型是否已注册
print("检查模型是否已注册:")
print(f"  is_model('simple_linear_net'): {is_model('simple_linear_net')}")
# print(f"  is_model('simple_linear_net_small'): {is_model('simple_linear_net_small')}")

# 列出所有包含 'simple' 的模型
print("\n查找包含 'simple' 的模型:")
simple_models = list_models('simple*')
for model_name in simple_models:
    print(f"  - {model_name}")

## 7. 使用 create_model 创建模型

现在我们可以像使用其他 timm 模型一样使用我们的自定义模型了！

我们有三种方式创建模型：
1. 使用 `simple_linear_net` - 默认配置
2. 使用 `simple_linear_net_small` - 小尺寸配置（独立注册的模型）
3. 使用 `simple_linear_net.small` - 通过标签访问小尺寸配置
4. 自定义参数覆盖配置

In [ ]:
# 方式1: 创建默认配置的模型
model1 = create_model('simple_linear_net')
print("模型1 (默认配置 - simple_linear_net):")
print(f"  输入尺寸: {model1.input_size}")
print(f"  类别数: {model1.num_classes}")
print(f"  特征维度: {model1.num_features}")
print(f"  参数量: {sum(p.numel() for p in model1.parameters()):,}")

# # 方式2: 使用独立注册的小尺寸模型
# model2 = create_model('simple_linear_net_small')
# print("\n模型2 (独立注册的小尺寸模型 - simple_linear_net_small):")
# print(f"  输入尺寸: {model2.input_size}")
# print(f"  类别数: {model2.num_classes}")
# print(f"  特征维度: {model2.num_features}")
# print(f"  参数量: {sum(p.numel() for p in model2.parameters()):,}")

# 方式3: 通过标签访问小尺寸配置（与方式2等效）
model3 = create_model('simple_linear_net.small')
print("\n模型3 (通过标签访问 - simple_linear_net.small):")
print(f"  输入尺寸: {model3.input_size}")
print(f"  类别数: {model3.num_classes}")
print(f"  特征维度: {model3.num_features}")
print(f"  参数量: {sum(p.numel() for p in model3.parameters()):,}")

# 方式4: 自定义参数覆盖配置
model4 = create_model(
    'simple_linear_net',
    num_classes=100,
    input_size=(3, 128, 128),
    drop_rate=0.2
)
print("\n模型4 (自定义参数):")
print(f"  输入尺寸: {model4.input_size}")
print(f"  类别数: {model4.num_classes}")
print(f"  Dropout率: {model4.drop_rate}")
print(f"  特征维度: {model4.num_features}")
print(f"  参数量: {sum(p.numel() for p in model4.parameters()):,}")

## 8. 测试模型前向传播

In [ ]:
# 创建随机输入
batch_size = 4
x = torch.randn(batch_size, 3, 224, 224)

# 前向传播
model = create_model('simple_linear_net')
model.eval()

with torch.no_grad():
    output = model(x)

print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"输出范围: [{output.min().item():.4f}, {output.max().item():.4f}]")

# 测试 softmax 输出
probs = torch.softmax(output, dim=1)
print(f"\n概率和: {probs.sum(dim=1)}")
print(f"最大概率: {probs.max(dim=1).values}")

## 9. 测试 timm 标准接口

In [ ]:
model = create_model('simple_linear_net', num_classes=1000)

# 测试 get_classifier
classifier = model.get_classifier()
print(f"分类器: {classifier}")
print(f"分类器输入维度: {classifier.in_features}")
print(f"分类器输出维度: {classifier.out_features}")

# 测试 reset_classifier
print("\n重置分类器为 10 类...")
model.reset_classifier(10)
classifier = model.get_classifier()
print(f"新分类器: {classifier}")
print(f"新分类器输出维度: {classifier.out_features}")

# 验证输出形状
with torch.no_grad():
    output = model(torch.randn(2, 3, 224, 224))
print(f"\n新输出形状: {output.shape}")

## 10. 模型结构可视化

In [ ]:
model = create_model('simple_linear_net', num_classes=10)
print("模型结构:")
print(model)

print("\n模型参数详情:")
for name, param in model.named_parameters():
    print(f"  {name:20s} - shape: {str(param.shape):20s} - 参数量: {param.numel():,}")

## 11. 完整训练示例（MNIST）

演示如何在实际数据集上训练我们的自定义模型。

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.optim as optim

# 准备数据
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(num_output_channels=3),  # 转换为3通道
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 加载 MNIST 数据集（如果不存在会自动下载）
try:
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    # 创建模型
    model = create_model(
        'simple_linear_net',
        num_classes=10,
        input_size=(3, 28, 28),
        drop_rate=0.2
    )
    
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # 训练函数
    def train_epoch(model, loader, criterion, optimizer, device):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch_idx, (data, target) in enumerate(loader):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
            
            if batch_idx % 100 == 0:
                print(f'  Batch {batch_idx}/{len(loader)}, Loss: {loss.item():.4f}')
        
        return total_loss / len(loader), 100. * correct / total
    
    # 测试函数
    def test(model, loader, criterion, device):
        model.eval()
        total_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for data, target in loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                
                total_loss += loss.item()
                pred = output.argmax(dim=1)
                correct += pred.eq(target).sum().item()
                total += target.size(0)
        
        return total_loss / len(loader), 100. * correct / total
    
    # 训练模型
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    print(f"\n使用设备: {device}")
    print("开始训练...\n")
    
    num_epochs = 3
    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = test(model, test_loader, criterion, device)
        
        print(f"  训练 - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
        print(f"  测试 - Loss: {test_loss:.4f}, Acc: {test_acc:.2f}%\n")
    
    print("训练完成!")
    
except Exception as e:
    print(f"训练示例跳过（可能是因为网络问题或数据集下载失败）: {e}")
    print("这不影响模型注册功能的演示。")

## 总结

通过这个教程，我们学习了：

1. ✅ 如何定义一个符合 timm 规范的自定义模型类
2. ✅ 如何使用 `generate_default_cfgs` 创建模型配置
3. ✅ 如何使用 `@register_model` 装饰器注册模型
4. ✅ 如何通过 `create_model()` 创建自定义模型
5. ✅ 如何实现 timm 的标准接口（`get_classifier`, `reset_classifier`）
6. ✅ 如何在实际数据集上训练自定义模型

### 关键要点

- **模型类**需要实现 `forward_features`, `forward_head`, `get_classifier`, `reset_classifier` 等方法
- **配置字典**使用 `generate_default_cfgs` 生成，支持多个配置标签
- **注册装饰器** `@register_model` 将模型添加到全局注册表
- **创建函数**使用 `build_model_with_cfg` 辅助函数处理配置和权重加载

### 扩展建议

你可以基于这个示例：
- 添加更复杂的网络结构（卷积层、注意力机制等）
- 添加预训练权重支持（设置 `url` 或 `hf_hub_id`）
- 实现更多 timm 标准接口（如 `forward_intermediates`）
- 添加模型变体（不同深度、宽度等）